In [ ]:
!pip install -q langchain_community
!pip install -q faiss-cpu
!pip install -q langchain-huggingface
!pip install -q langchain_openai
!pip install unstructured openpyxl
!pip install msoffcrypto-tool

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.8/48.8 kB 2.2 MB/s eta 0:00:00


# Importing libraries

In [ ]:
from langchain_community.document_loaders import TextLoader
from langchain_community.document_loaders import UnstructuredExcelLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS
from langchain_core.prompts import PromptTemplate
import os
from langchain_openai import ChatOpenAI
from langchain.tools import tool
from langchain.agents import create_agent

In [ ]:
import os

file_path = "Claims_dataset_RAG_project.xlsx"

print(os.getcwd())
print(os.path.exists(file_path))

/content
True


# Load Text Document

In [ ]:
loader = UnstructuredExcelLoader("Claims_dataset_RAG_project.xlsx")
documents = loader.load()

print("Number of documents loaded:", len(documents))
print("Content:", documents[0].page_content)
print("Metadata:", documents[0].metadata)

Number of documents loaded: 1
Content: Claim_ID Patient_ID Payer Provider Date_of_Service CPT_Code ICD_Code Billed_Amount Allowed_Amount Payment_Amount Denial_Code Denial_Reason Appeal_Status Documentation_Notes Appeal_Instructions CLM10001 PAT5001 BCBS Dr. Smith 2025-03-12 00:00:00 99214 E11.9 225 0 0 CO50 Not medically necessary Pending Progress note indicates routine diabetes follow-up; moderate complexity not documented Submit additional clinical documentation showing medical necessity for CPT 99214 CLM10002 PAT5002 Aetna Dr. Lee 2025-03-15 00:00:00 93000 I10 150 0 0 CO16 Missing documentation Submitted ECG performed but report missing in claim submission Upload missing ECG report for appeal CLM10003 PAT5003 UHC Dr. Patel 2025-03-20 00:00:00 70553 R51.9 1200 1200 1200 Approved MRI documentation complete; prior authorization obtained CLM10004 PAT5004 Cigna Dr. Adams 2025-03-22 00:00:00 99213 J06.9 175 175 175 Approved Routine office visit with complete documentation CLM10005 PAT5005

# Split the Document

In [ ]:
import pandas as pd
from langchain_core.documents import Document
df = pd.read_excel("Claims_dataset_RAG_project.xlsx")

documents = []

for index, row in df.iterrows():
    content = row.to_string()
    documents.append(
        Document(
            page_content=content,
            metadata={"row_number": index}
        )
    )


In [ ]:
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=50
)

chunks = text_splitter.split_documents(documents)

print("Original documents:", len(documents))
print("Chunks created:", len(chunks))
print(chunks[0].page_content)
print(chunks[0].metadata)

Original documents: 43
Chunks created: 119
Claim_ID                                                        CLM10001
Patient_ID                                                       PAT5001
Payer                                                               BCBS
Provider                                                       Dr. Smith
Date_of_Service                                      2025-03-12 00:00:00
CPT_Code                                                           99214
{'row_number': 0}


# Initialize the Text Embedding

In [ ]:
embeddings = HuggingFaceEmbeddings(
    model_name = "sentence-transformers/all-mpnet-base-v2")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/363 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [ ]:
vectorstore = FAISS.from_documents(chunks, embeddings)

# Create A Retriever

In [ ]:
retriever = vectorstore.as_retriever()

# Creating different tools for the Agent

In [ ]:
@tool
def docu_search (query: str) -> str:
  """
  search claims dataset RAG Project for relevant information
  use this tool when answering questions about appeal instructions, denial reason, Documentation notes, Payer,
  Provider, CPT code, ICD code, Billed Amount, Payment Amount, Patient ID.
  """
  docs=  retriever.invoke(query)
  context_parts =[]
  for doc in docs:
    context_parts.append(doc.page_content)
  return "\n\n".join(context_parts)

@tool
def get_payment_amount_stats(dummy_input: str = "") -> str:
    """
    Reads the loaded DataFrame and returns the minimum and maximum
    values from the Payment_Amount column. Pass any string as dummy_input.
    """
    if "Payment_Amount" not in df.columns:
        return (
            f"Column 'Payment_Amount' not found. "
            f"Available columns: {list(df.columns)}"
        )

    billed = pd.to_numeric(df["Payment_Amount"], errors="coerce").dropna()

    if billed.empty:
        return "The 'Payment_Amount' column has no valid numeric values."

    min_val = billed.min()
    max_val = billed.max()
    return (
        f"Payment_Amount stats — "
        f"Minimum: {min_val} | Maximum: {max_val}"
    )

@tool
def get_payer_max_payment(dummy_input: str = "") -> str:
    """
    Reads the loaded DataFrame and returns the Payer name
    who has the maximum Payment_Amount. Pass any string as dummy_input.
    """
    if "Payment_Amount" not in df.columns or "Payer" not in df.columns:
        return (
            f"Required columns not found. "

        )

    df_clean = df.copy()
    df_clean["Payment_Amount"] = pd.to_numeric(df_clean["Payment_Amount"], errors="coerce")
    df_clean = df_clean.dropna(subset=["Payment_Amount"])

    if df_clean.empty:
        return "The 'Payment_Amount' column has no valid numeric values."

    max_row = df_clean.loc[df_clean["Payment_Amount"].idxmax()]
    return (
        f"Payer with maximum Payment_Amount: {max_row['Payer']} "
        f"| Amount: {max_row['Payment_Amount']}"
    )
@tool
def get_payer_min_payment(dummy_input: str = "") -> str:
    """
    Reads the loaded DataFrame and returns the Payer name
    who has the minimum Payment_Amount. Pass any string as dummy_input.
    """
    if "Payment_Amount" not in df.columns or "Payer" not in df.columns:
        return (
            f"Required columns not found. "

        )

    df_clean = df.copy()
    df_clean["Payment_Amount"] = pd.to_numeric(df_clean["Payment_Amount"], errors="coerce")
    df_clean = df_clean.dropna(subset=["Payment_Amount"])

    if df_clean.empty:
        return "The 'Payment_Amount' column has no valid numeric values."

    min_row = df_clean.loc[df_clean["Payment_Amount"].idxmin()]
    return (
        f"Payer with minimum Payment_Amount: {min_row['Payer']} "
        f"| Amount: {min_row['Payment_Amount']}"
    )

# Initializing LLM

In [ ]:
os.environ["OPENAI_API_KEY"] = "####****"
llm = ChatOpenAI(model_name = "gpt-3.5-turbo", temperature = 0.0)

# Create the Agent

In [ ]:
agent = create_agent(
    model= llm,
    tools = [docu_search, get_payment_amount_stats, get_payer_max_payment, get_payer_min_payment],
    system_prompt= ( """
    You are an assistant for question answering tasks. Use the tools provided to you to asnwer the questions.
    if you dont know the answer just say you dont know.
    Use one sentence and keep the answer concise.
    """
),
)

# Ask the question

In [ ]:
result = agent.invoke(
    {
        "messages": [
        {"role": "user", "content": "What is the minimum and maximum Payment Amount?"

        } ]
    }
)
print(result["messages"][-1].content)

The minimum Payment Amount is 0.0, and the maximum Payment Amount is 1200.0.


In [ ]:
result = agent.invoke(
    {
        "messages": [
        {"role": "user", "content": "What is the most common Denial reason?"

        } ]
    }
)
print(result["messages"][-1].content)

The most common Denial reason is "Missing documentation."


In [ ]:
result = agent.invoke(
    {
        "messages": [
        {"role": "user", "content": "What payer paid the most?"

        } ]
    }
)
print(result["messages"][-1].content)

The payer that paid the most is UHC with a payment amount of $1200.
